### Compute power, sensitivity and PPV for results obtained over groups of datasets

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from tqdm import tqdm

import folium
import branca.colormap as cm

In [ ]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)
# display(dict_candidates)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

For each candidate, retrieve the grid and subset of cell it refers to.

In [ ]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Retrieve the flattened list of object IDs associated with the candidates.
flattened_list_candidates = dict_candidates['flat_ids']

# Compute the starting pos, ending pos, and number of objects of each candidate.
start_pos_candidates = dict_candidates['start_pos'][:-1]
end_pos_candidates = dict_candidates['start_pos'][1:]
num_objs_candidates = np.diff(dict_candidates['start_pos'])

# Some DEBUG.
# print(len(start_pos_candidates), len(end_pos_candidates), len(num_objs_candidates))
# print(start_pos_candidates[-4], end_pos_candidates[-5])

In [ ]:
# For each candidate, find out the resolution of the grid it comes from.
num_candidates = start_pos_candidates.size
candidates_grid_res = np.empty(num_candidates, dtype=np.uint32)
count = 0
for grid in grid_info:
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=np.uint32)
    grid_id[0] = grid[1]
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    candidates_grid_res[count : count + num_els_grid] = grid_id

    count += num_els_grid

# Create the list of grid resultions used during the assessment.
# NOTE: the zero resolution is used to consider the candidates from ALL the grids.
list_grid_res = [0] + np.unique(candidates_grid_res).tolist()


display(list_grid_res)
display(candidates_grid_res)

Now process the results...

In [ ]:
path_unfair_datasets = './experiments/num_objects/'
list_files_datasets = [f for f in Path(path_unfair_datasets).iterdir() if (f.is_file() and 'results' not in f.name)]
list_files_results = [f for f in Path(path_unfair_datasets).iterdir() if (f.is_file() and 'results' in f.name)]


# Read the results computed over a given group of datasets from disk.
idx_tuple_list_objs = 1
for path_results, path_datasets in zip(list_files_results, list_files_datasets):
    
    # Read the results computed over a group of datasets.
    with open(path_results, "rb") as f: set_results = pickle.load(f)
    # display(set_results)

    # Compute the number of datasets in this group.
    num_datasets_group = len(set_results['idx_candidates'])

    # 1 - Determine the statistical power of the approach considering all the candidates (and thus grids) used.
    stat_power = np.sum([len(result) != 0 for result in set_results['idx_candidates']])

    # Read the unfair datasets (needed to retrieve the list of object IDs belonging to the unfair hotspots).
    with open(path_datasets, "rb") as f:
        set_datasets = pickle.load(f)

    # 2 - Compute the sensitivity and PPV over this group of datasets.
    sum_sensitivity, sum_ppv = 0., 0.
    for idx_dataset in tqdm(range(num_datasets_group)) :

        # Retrieve the lists of object IDs associated with the various hotspots in this unfair dataset.
        # Each hotspot's list is a 1D numpy array, so we need to concatenate these arrays.
        # Guarantee also that the IDs in the final list are unique.
        set_unfair_obj_ids = np.unique(np.concatenate(set_datasets['data'][idx_dataset][idx_tuple_list_objs]))

        # Retrieve the extreme candidates detected for this dataset.
        set_detected_candidates = set_results['idx_candidates'][idx_dataset]


        # Retrieve the IDs of the objects associated with the extreme candidates found by the assessment
        # approach.
        list_detected_cand_objs = \
            [flattened_list_candidates[start_pos_candidates[i] : end_pos_candidates[i]] for i in set_detected_candidates]
        list_detected_cand_objs = np.unique(np.concatenate(list_detected_cand_objs)) if list_detected_cand_objs else np.empty(0)

        # Compute the set intersection between the set of true unfair object IDs and the set of object IDs associated with
        # the candidates deemed 'extreme' by the assessment approach.
        intersect_obj_ids = np.intersect1d(set_unfair_obj_ids, list_detected_cand_objs)


        # Compute sensitivity and PPV. Recall:
        # - sensitivity:  measures the fraction of truly affected objects that are correctly flagged by an assessment approach
        # - ppv: measures the fraction of objects flagged by an assessment approach that truly belong to the true set.
        sensitivity = intersect_obj_ids.size / set_unfair_obj_ids.size
        ppv = intersect_obj_ids.size / list_detected_cand_objs.size if list_detected_cand_objs.size else 0.
        sum_sensitivity += sensitivity
        sum_ppv += ppv

        # DEBUG...
        #if lists_cand_objs.size :
        #    print(f"DEBUG: Set true unfair obj IDs: {set_unfair_obj_ids}")
        #    print(f"DEBUG: Set detected candidates: {set_detected_candidates}")
        #    print(f"DEBUG: List detected candidates' obj IDs: {lists_cand_objs}")
        #    print(f"DEBUG: sensitivity and PPV for this unfair dataset: {sensitivity}, {ppv}")
        #    break


    sum_sensitivity = sum_sensitivity / stat_power
    sum_ppv = sum_ppv / stat_power
    stat_power = stat_power / num_datasets_group
    print(f"DEBUG: Power, sensitivity and PPV for {path_results.name}: {stat_power}, {sum_sensitivity}, {sum_ppv}")
    # break